<a href="https://colab.research.google.com/github/Rithusravya/Emasters_Group-2_CapstoneProject/blob/main/check_point_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers accelerate torch sentencepiece
!pip install datasets evaluate nltk sqlparse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 911.6 kB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [5]:
small_model = "Salesforce/codegen-350M-multi"

# 1. Load the Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(small_model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(small_model)
model.config.pad_token_id = tokenizer.pad_token_id

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  797MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  797MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/165 [00:00<?, ?it/s]

[transformers] CodeGenForCausalLM LOAD REPORT from: Salesforce/codegen-350M-multi
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...19}.attn.causal_mask | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


CodeGenForCausalLM(
  (transformer): CodeGenModel(
    (wte): Embedding(51200, 1024)
    (drop): Dropout(p=0.0, inplace=False)
    (h): ModuleList(
      (0-19): 20 x CodeGenBlock(
        (ln_1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (attn): CodeGenAttention(
          (attn_dropout): Dropout(p=0.0, inplace=False)
          (resid_dropout): Dropout(p=0.0, inplace=False)
          (qkv_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=False)
        )
        (mlp): CodeGenMLP(
          (fc_in): Linear(in_features=1024, out_features=4096, bias=True)
          (fc_out): Linear(in_features=4096, out_features=1024, bias=True)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=1024, out_features=51200, bias=True)
)

In [6]:
# 2. Define the Text Generation Function
def generate_text(prompt, max_tokens=200):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.2,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    result = tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

    return result

In [7]:
# 3. Define the Documentation Function
def generate_documentation(code):
    prompt = f"""Generate documentation for this Python code:

{code}

Documentation:
"""
    return generate_text(prompt, max_tokens=200)

In [8]:
prompt = "Write a Python program to find the factorial of a number."

In [9]:
print("--- Generating Code ---")
generated_program = generate_text(prompt)
print(generated_program)

print("\n" + "=" * 40 + "\n")

print("--- Generating Documentation ---")
documentation = generate_documentation(generated_program)
print(documentation)

--- Generating Code ---
Write a Python program to find the factorial of a number.

# Example 1:

# Input: 3
# Output: 5
# Explanation: The factorial of 3 is 5.

# Example 2:

# Input: 7
# Output: 15
# Explanation: The factorial of 7 is 15.

# Note:

# 1 <= n <= 100
# 1 <= n <= 10000
# 1 <= n <= 1000000

class Solution(object):
    def factorial(self, n):
        """
        :type n: int
        :rtype: int
        """
        if n <= 0:
            return 0
        if n == 1:
            return 1
        if n == 2:
            return 2
        if n == 3:
            return 3
        if n == 4:
            return 5
        if n == 5:
            return 15
        if n == 6:
            


--- Generating Documentation ---
Generate documentation for this Python code:

Write a Python program to find the factorial of a number.

# Example 1:

# Input: 3
# Output: 5
# Explanation: The factorial of 3 is 5.

# Example 2:

# Input: 7
# Output: 15
# Explanation: The factorial of 7 is 15.

# Note:

In [10]:
import time
import psutil
import os

start = time.time()

generated_text = generate_text(prompt)

end = time.time()

inference_time = end - start

tokens = len(tokenizer.encode(generated_text))
tokens_per_sec = tokens / inference_time

memory = psutil.Process(os.getpid()).memory_info().rss / (1024**2)

print(f"Inference Time : {inference_time:.2f} sec")
print(f"Tokens Generated : {tokens}")
print(f"Speed : {tokens_per_sec:.2f} tokens/sec")
print(f"Memory Usage : {memory:.2f} MB")

Inference Time : 33.46 sec
Tokens Generated : 198
Speed : 5.92 tokens/sec
Memory Usage : 1463.01 MB
